# GPU 长什么样：lspci 与 PCIe 拓扑

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-02-gpu-on-pcie.ipynb)

从系统总线视角看 GPU。用 lspci、nvidia-smi -q 和 /proc/cpuinfo 把 GPU 和它在主板上的位置、NUMA 归属联系起来，为后面理解多卡、PCIe 带宽和拓扑调度打基础。

本 notebook 配合教程章节 [GPU 长什么样：lspci 与 PCIe 拓扑](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 里 `nvidia-smi` 显示了一行 `00000000:00:04.0`，那是 GPU 的 **PCIe 总线地址**。在单卡 Colab 里它只是个数字，但在多卡服务器里，GPU 挂在哪个 NUMA 节点、走哪条 PCIe Switch、和谁共享带宽，直接决定了训练和推理的性能。Kubernetes 的拓扑调度、HAMi 的设备分配，本质上都在围绕这些信息做决策。本 Lab 用最小命令把这些概念落地。

先建立整体图景：CPU 和离它最近的内存组成一个 **NUMA 节点**；CPU 里的 PCIe 根复合体通过一条 **PCIe 链路**连到 GPU，链路的代际和 lane 数决定了 **带宽** 上限；`lspci` 等命令则分别观察这条通路上的不同部分。下图把这几个概念和本 Lab 用到的命令对应起来：

![一块 GPU 在机器里的位置：NUMA、PCIe、带宽与查看命令](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/pcie-concept-map.svg)

## 第 1 步：用 lspci 看总线

Colab 镜像没有预装 `lspci`，直接运行会报 `lspci: command not found`。先用 `apt-get` 安装它所在的 `pciutils` 包（不需要先 `apt-get update`，约 10 秒），再过滤出 NVIDIA 设备：

In [ ]:
!apt-get install -y -qq pciutils > /dev/null
!lspci | grep -i nvidia

预期输出类似：

```text
00:04.0 3D controller: NVIDIA Corporation TU104GL [Tesla T4] (rev a1)
```

读法：

- `00:04.0` = **总线 : 设备 . 功能**（bus:device.function），和 `nvidia-smi` 里的 `00:04.0` 对应；
- `3D controller` = 设备类别（数据中心卡常显示为 `3D controller` 而非 `VGA compatible controller`，因为它没有显示输出）；
- `TU104GL` = 芯片代号（Turing）。

## 第 2 步：用 nvidia-smi 查 PCIe 链路

In [ ]:
!nvidia-smi --query-gpu=pci.bus_id,pcie.link.gen.max,pcie.link.gen.current,pcie.link.width.max,pcie.link.width.current --format=csv

预期看到：

```text
pci.bus_id, pcie.link.gen.max, pcie.link.gen.current, pcie.link.width.max, pcie.link.width.current
00000000:00:04.0, 3, 1, 16, 16
```

关键字段：

- `pcie.link.gen.max: 3` = 设备支持的最高代际 PCIe **Gen3**（T4 上限）；
- `pcie.link.gen.current: 1` = **当前**运行在 Gen1。这不是故障：GPU 空载时会把链路降速省电，有数据传输时自动升回 Gen3；
- `pcie.link.width.max: 16` = 16 条 lane，即 **x16**。

想看完整的 PCI 小节，可以从 `nvidia-smi -q` 的完整报告里截取。注意 `-d` 参数不接受 `PCI`（580 驱动会报 `Failed to parse --display/-d flags`），所以用 `sed` 按小节标题截取：

In [ ]:
!nvidia-smi -q | sed -n "/^    PCI$/,/^    Fan Speed/p"

它会把上面这些字段按 `GPU Link Info → PCIe Generation / Link Width` 的层级展开，还会列出 Device Id（T4 是 `0x1EB810DE`，`10DE` 是 NVIDIA 的 vendor id）。

## 第 3 步：算一下理论带宽

PCIe 是全双工的，每条 lane 单向带宽约：



| 代际 | 单 lane 单向 (GB/s) | x16 单向 (GB/s) | x16 双向合计 (GB/s) |
| :--- | :--- | :--- | :--- |
| Gen3 | ~0.985 | ~15.75 | ~31.5 |
| Gen4 | ~1.969 | ~31.5 | ~63 |
| Gen5 | ~3.938 | ~63 | ~126 |



T4 在 Gen3 x16 下，CPU→GPU 单向理论峰值约 **15.75 GB/s**（2026-11-07 在 Colab 上用 PyTorch 从 pinned 内存拷贝实测约 12.4 GB/s；普通 pageable 内存只有约 5 GB/s）。对比 T4 的显存带宽 **320 GB/s**：**数据一旦进了显存，访问速度比从 CPU 搬快 20 倍以上**。这就是为什么 GPU 编程要尽量减少“CPU↔GPU 来回搬数据”（[Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 会再遇到）。

把这些数字按同一比例画出来，差距更直观：

![带宽阶梯：T4 显存带宽与各代 PCIe 理论带宽、Colab 实测拷贝带宽的对比](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/bandwidth-ladder.svg)

## 第 4 步：NUMA 归属（理解它，跳过实现）

In [ ]:
!lscpu | grep -E "^(Socket|NUMA)"
!nvidia-smi topo -m

在 Colab T4 上的实际输出如下（2026-11-07 实测，`topo -m` 末尾的 Legend 省略）：

```text
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0,1
    GPU0	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	0-1	0		N/A
```

前三行来自 `lscpu`，描述 CPU 侧的布局；后两行是 `nvidia-smi topo -m` 的拓扑矩阵，描述 GPU 挂在哪里。逐项含义如下：



| 字段 | Colab 上的值 | 含义 |
| :--- | :--- | :--- |
| `Socket(s)` | `1` | 物理 CPU 插槽数。多路服务器常见 `2`，每个插槽通常是一个 NUMA 节点 |
| `NUMA node(s)` | `1` | NUMA 节点数。只有 1 个时，所有内存对所有 CPU 一样近，不存在“远端内存” |
| `NUMA node0 CPU(s)` | `0,1` | 节点 0 包含的逻辑 CPU 编号。Colab 只分到 2 个 vCPU |
| 矩阵的行列 `GPU0` | `X` | 矩阵记录 GPU 两两之间的连接类型；对角线是自己，用 `X` 表示。单卡时矩阵只有这一格 |
| `CPU Affinity` | `0-1` | 离这张 GPU 最近的 CPU 核。给 GPU 喂数据的进程最好绑在这些核上 |
| `NUMA Affinity` | `0` | GPU 所在 PCIe 根复合体属于哪个 NUMA 节点，即 GPU“挂在”哪组 CPU 和内存下面 |
| `GPU NUMA ID` | `N/A` | 只有 GPU 显存本身被暴露成 NUMA 节点时才有值（如 Grace Hopper 这类 CPU-GPU 一致性内存平台）；普通 PCIe 卡为 `N/A` |



单卡时矩阵里只有 `X`，所以 Legend 里列出的连接类型都用不上。到了多卡服务器，矩阵的非对角格会填上这些缩写，表示两张卡之间的数据要走哪条路。下表按从近到远排列：



| 缩写 | 路径 | 通俗理解 |
| :--- | :--- | :--- |
| `NV#` | 走 # 条捆绑的 NVLink | GPU 直连，最快，不经过 PCIe |
| `PIX` | 最多经过一个 PCIe 桥 | 两张卡挂在同一个 PCIe Switch 下 |
| `PXB` | 经过多个 PCIe 桥，但不经过 Host Bridge | 跨了几级 Switch，仍不用绕到 CPU |
| `PHB` | 经过 PCIe Host Bridge（通常就是 CPU） | 数据要绕到 CPU 的根复合体再下来 |
| `NODE` | 经过同一 NUMA 节点内多个 Host Bridge 之间的互连 | 同一颗 CPU 下，但挂在不同根端口 |
| `SYS` | 经过 NUMA 节点之间的 SMP 互连（如 Intel QPI/UPI） | 跨 CPU 插槽，最慢 |



越靠上的连接，GPU 之间点对点通信（如 NCCL 的 all-reduce）越快。Kubernetes 的拓扑感知调度要做的，就是尽量把同一个任务的多张卡分在 `NV#` / `PIX` 这类“近”的组合里，并把 Pod 的 CPU 绑在 `CPU Affinity` 指示的核上。在多卡服务器上，不同 GPU 会落到不同 NUMA 节点：

![多卡服务器上的 PCIe / NUMA 拓扑：两个 CPU 各挂一个 PCIe Switch 和两张 GPU，GPU 0 与 GPU 1 之间有 NVLink](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/pcie-numa-topology.svg)

> 这正是 Kubernetes [Device Plugin / DRA 边界](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/foundations/device-plugin-vs-dra/) 里“拓扑约束无法用整数表达”的现实来源：调度器不只关心“有几张卡”，还关心“卡和 CPU 谁更近”。

## 验证

完成后你应该能：

- 用 `lspci` 找到 GPU 的 bus:device.function；
- 把它和 `nvidia-smi` 的 `Bus Id` 对应起来；
- 说出 T4 的 PCIe 代际、链路宽度、理论带宽，以及为什么空载时 `current` 会低于 `max`；
- 解释为什么“数据进显存后比从 CPU 搬快 20 倍”。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `lspci: command not found` | 镜像没有 `pciutils` | `!apt-get install -y -qq pciutils` |
| `lspci` 里看不到 NVIDIA | Runtime 还是 CPU | 切到 T4 GPU runtime，重连 |
| `pcie.link.gen.current` 是 `1` | 空载省电降速，正常 | 跑一个显存拷贝后再查，会升到 `3` |
| `pcie.link.width.current` 显示 `4` / `8` 而非 `16` | 虚拟化/共享带宽或降速 | 一般不影响学习；多见于被 vGPU/分片的环境 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。